# Dataset Merge & Deduplication
**Pipeline:** Load 3 CSVs → Merge → Exact match dedup → BLEU-based near-dedup → Build turns dataset

Deduplication happens at **two levels**:
1. **Conversation level** — fingerprint via first-3-query exact match
2. **Turn level** — exact match on user_query, then BLEU similarity to catch paraphrased near-dupes

In [1]:
# ── Install deps (run once) ──────────────────────────────────────────────────
!pip install nltk --quiet

In [2]:
import pandas as pd
import json
import ast
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
from collections import defaultdict
import re

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✓ Imports done")

✓ Imports done


## 1. Load & Merge

In [5]:
# ── CONFIG: point these at your files ────────────────────────────────────────
FILES = {
    "old":     "conversations -old.csv",
    "llama17b": "conversations -llama 17b.csv",
    "main":    "conversations.csv",
}

# Priority order for dedup (first = preferred to keep)
PRIORITY = ["old", "llama17b", "main"]

# BLEU threshold for near-duplicate turns (0–1, higher = stricter)
BLEU_THRESHOLD = 0.85
# ─────────────────────────────────────────────────────────────────────────────

dfs = []
for key in PRIORITY:
    df = pd.read_csv(FILES[key])
    df["_source"] = key
    df["_priority"] = PRIORITY.index(key)
    dfs.append(df)
    print(f"  {key:10s} → {len(df):4d} rows")

all_df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal merged: {len(all_df)} rows")

  old        →   98 rows
  llama17b   →  237 rows
  main       →    9 rows

Total merged: 344 rows


## 2. Parse list-type columns

In [6]:
def safe_parse(s):
    """Parse JSON or Python-literal list strings robustly."""
    if isinstance(s, list):
        return s
    try:
        return json.loads(s)
    except Exception:
        pass
    try:
        return ast.literal_eval(s)
    except Exception:
        return []

LIST_COLS = ["user_queries", "expanded_queries", "named_entities", "needs_expansion"]
for col in LIST_COLS:
    all_df[f"_{col}"] = all_df[col].apply(safe_parse)

print("Sample user_queries parse:", all_df["_user_queries"].iloc[0][:2])
print("Sample needs_expansion parse:", all_df["_needs_expansion"].iloc[0][:3])

Sample user_queries parse: ['who is the 13th prime minister of india?', 'what were his major economic reforms?']
Sample needs_expansion parse: [False, True, True]


## 3. Conversation-level deduplication

**Method:** Fingerprint each conversation with its first 3 user queries (lowercased, stripped). Exact match → keep the highest-priority source version.

In [7]:
def conv_fingerprint(uq_list):
    return "|||" .join(q.strip().lower()[:100] for q in uq_list[:3])

all_df["_conv_fp"] = all_df["_user_queries"].apply(conv_fingerprint)

# Sort by priority so drop_duplicates keeps the best source
all_df = all_df.sort_values("_priority")

before = len(all_df)
conv_deduped = all_df.drop_duplicates(subset=["_conv_fp"], keep="first").copy()
after = len(conv_deduped)

print(f"Conversations before: {before}")
print(f"Conversations after:  {after}")
print(f"Removed (exact conv): {before - after}")

# Show which sources contributed
print("\nSource breakdown after dedup:")
print(conv_deduped["_source"].value_counts())

Conversations before: 344
Conversations after:  277
Removed (exact conv): 67

Source breakdown after dedup:
_source
llama17b    177
old          91
main          9
Name: count, dtype: int64


## 4. Build turns dataset (explode conversations → individual turns)

In [8]:
rows = []
skipped = 0

for _, row in conv_deduped.iterrows():
    uqs   = row["_user_queries"]
    eqs   = row["_expanded_queries"]
    nes   = row["_named_entities"]
    needs = row["_needs_expansion"]

    if not uqs:
        skipped += 1
        continue

    n = min(len(uqs), len(eqs), len(nes), len(needs))
    for i in range(n):
        rows.append({
            "conversation_id": row["conversation_id"],
            "topic_l1":        row["topic_l1"],
            "topic_l2":        row["topic_l2"],
            "label":           f"{row['topic_l1']}_{row['topic_l2']}",
            "turn_index":      i + 1,
            "user_query":      uqs[i],
            "expanded_query":  eqs[i],
            "named_entities":  json.dumps(nes[i]) if isinstance(nes[i], list) else str(nes[i]),
            "needs_expansion": needs[i],
            "_source":         row["_source"],
        })

turns_df = pd.DataFrame(rows)
print(f"Total turns exploded: {len(turns_df)}")
print(f"Conversations skipped (empty): {skipped}")

Total turns exploded: 2770
Conversations skipped (empty): 0


## 5. Turn-level deduplication

### Step A — Exact string match

In [9]:
turns_df["_uq_norm"] = turns_df["user_query"].str.strip().str.lower()

before_exact = len(turns_df)
turns_exact = turns_df.drop_duplicates(subset=["_uq_norm"], keep="first").copy()
after_exact = len(turns_exact)

print(f"Turns before exact dedup: {before_exact}")
print(f"Turns after  exact dedup: {after_exact}")
print(f"Exact duplicates removed: {before_exact - after_exact}")

Turns before exact dedup: 2770
Turns after  exact dedup: 2047
Exact duplicates removed: 723


### Step B — BLEU-based near-duplicate detection

For each turn, compute BLEU against a running set of "kept" queries (grouped by topic label to keep it fast). If BLEU ≥ threshold → skip as near-duplicate.

In [10]:
smoother = SmoothingFunction().method1

def bleu_sim(hyp_tokens, ref_tokens):
    """Compute BLEU of hyp against a single ref."""
    return sentence_bleu(
        [ref_tokens], hyp_tokens,
        weights=(0.5, 0.5, 0, 0),   # bigram BLEU — good balance for short queries
        smoothing_function=smoother
    )

def tokenize(text):
    return word_tokenize(text.lower())

print(f"Running BLEU dedup with threshold={BLEU_THRESHOLD} ...")
print("(Grouped by label for speed — only compares within same topic)\n")

kept_mask = [True] * len(turns_exact)   # start: keep everything
turns_list = turns_exact.reset_index(drop=True)

# Group by label — no point comparing Politics_India with Sports_Cricket
label_buckets = defaultdict(list)   # label → list of (idx, tokens)

near_dup_pairs = []   # for inspection

for idx, row in turns_list.iterrows():
    tokens = tokenize(row["_uq_norm"])
    label  = row["label"]
    is_dup = False

    for (kept_idx, kept_tokens) in label_buckets[label]:
        score = bleu_sim(tokens, kept_tokens)
        if score >= BLEU_THRESHOLD:
            kept_mask[idx] = False
            near_dup_pairs.append({
                "removed_idx":   idx,
                "removed_query": row["user_query"],
                "kept_idx":      kept_idx,
                "kept_query":    turns_list.loc[kept_idx, "user_query"],
                "bleu":          round(score, 4),
                "label":         label,
            })
            is_dup = True
            break   # no need to check more

    if not is_dup:
        label_buckets[label].append((idx, tokens))

turns_bleu = turns_list[kept_mask].copy()
near_dup_df = pd.DataFrame(near_dup_pairs)

print(f"Turns after exact dedup:        {len(turns_list)}")
print(f"Near-duplicates found (BLEU≥{BLEU_THRESHOLD}): {len(near_dup_pairs)}")
print(f"Turns after BLEU dedup:         {len(turns_bleu)}")

Running BLEU dedup with threshold=0.85 ...
(Grouped by label for speed — only compares within same topic)

Turns after exact dedup:        2047
Near-duplicates found (BLEU≥0.85): 13
Turns after BLEU dedup:         2034


### Inspect near-duplicate pairs found by BLEU

In [11]:
if len(near_dup_df) > 0:
    print(f"Sample near-duplicate pairs (showing up to 10):")
    display(near_dup_df[["label", "bleu", "removed_query", "kept_query"]].head(10))
else:
    print("No near-duplicates found at this threshold.")

Sample near-duplicate pairs (showing up to 10):


,label,bleu,removed_query,kept_query
0,Health_Fitness,0.8528,"wait, give me a minute to grab a water bottle","wait, give me a sec to grab a water bottle"
1,Sports_Tennis,0.9293,"Give me a minute, I need to check something......","Give me a minute, I need to check something......"
2,History_World Wars,0.8819,And what about the war in the Pacific?,what about the war in the pacific?
3,Health_Nutrition,0.9131,"Wait a sec, I need to take a call.","wait a sec, I need to take a call. brb"
4,Politics_India,0.8771,"Back, so what were we saying about Manmohan Si...","back, so what were we saying about Manmohan Si..."
5,Politics_India,0.8825,who is the current prime minister of india,Who is the current Prime Minister of India?
6,Politics_USA,0.8819,and what about his role in the economy?,What about his role in the economy?
7,Sports_Cricket,0.8669,what is his role in the team,What is his role in the team?
8,Sports_Cricket,0.8825,back - how does he perform under pressure,back - how does he perform under pressure?
9,Politics_USA,0.8819,And what about his role in foreign policy?,What about his role in foreign policy?


## 6. Final stats & label distribution

In [12]:
final_turns = turns_bleu.drop(columns=["_uq_norm", "_source"]).reset_index(drop=True)

print("=" * 50)
print("FINAL TURNS DATASET")
print("=" * 50)
print(f"Total rows:          {len(final_turns)}")
print(f"Unique labels:       {final_turns['label'].nunique()}")
print(f"needs_expansion=True: {final_turns['needs_expansion'].sum()}")
print()
print("Label distribution:")
print(final_turns["label"].value_counts().to_string())

FINAL TURNS DATASET
Total rows:          2034
Unique labels:       20
needs_expansion=True: 1099

Label distribution:
label
Technology_AI              204
Politics_India             174
Politics_USA               171
Sports_Cricket             154
Entertainment_Hollywood    149
Sports_Tennis              143
Technology_Space           121
Entertainment_Bollywood    104
Politics_UK                102
Health_Fitness              95
Sports_Football             94
General_General             90
Technology_Gadgets          69
History_World Wars          65
Health_Mental Health        60
Geography_Countries         59
Sports_Olympics             58
Health_Nutrition            51
Geography_Cities            37
History_India               34


## 7. Save outputs

In [13]:
# ── Clean up merged conversations ────────────────────────────────────────────
drop_internal = [c for c in conv_deduped.columns if c.startswith("_")]
merged_conv = conv_deduped.drop(columns=drop_internal).reset_index(drop=True)

# Save
merged_conv.to_csv("merged_conversations.csv", index=False)
final_turns.to_csv("turns_dataset.csv", index=False)

if len(near_dup_df) > 0:
    near_dup_df.to_csv("near_duplicate_report.csv", index=False)
    print("✓ near_duplicate_report.csv saved")

print(f"✓ merged_conversations.csv  — {len(merged_conv)} conversations")
print(f"✓ turns_dataset.csv         — {len(final_turns)} turns (BERT-ready)")

✓ near_duplicate_report.csv saved
✓ merged_conversations.csv  — 277 conversations
✓ turns_dataset.csv         — 2034 turns (BERT-ready)
